# Математический маятник

## Базовый уровень

Решаем уравнение математического маятника в линейном приближении: $$\ddot x + \omega^2 x = 0.$$

Сделаем замену $v = \dot x$.

Тогда обыкновеннное дифференциальное уравнение второго порядка можно 
записать в виде системы обыкновенных дифференциальных уравнений порядка:

\begin{equation*}
\begin{cases}
\dot x = v\\
\dot v = -\omega^2 x.
\end{cases}
\end{equation*}

Фазовое состояние характеризуется переменной $(x, v)$.

Будем реализовывать следующую схему Эйлера для такой системы уравнений:

\begin{equation*}
\begin{cases}
x^{i+1} = x^i + v^i \Delta t\\
v^{i+1} = v^i - \omega^2 x^i \Delta t.
\end{cases}
\end{equation*}

И схему Рунге-Кутты 4-го порядка:

\begin{equation*}
\begin{cases}
x^{i+1} = x^i + \frac{\Delta t}{6} \left(k_{x1} + 2 k_{x2} + 2 k_{x3} + k_{x4}\right)\\
v^{i+1} = v^i + \frac{\Delta t}{6} \left(k_{v1} + 2 k_{v2} + 2 k_{v3} + k_{v4}\right),
\end{cases}
\end{equation*}

где $k_{x1} = v^{i}$,
$k_{x2} = v^{i} + \frac{\Delta t}{2} k_{v1}$,
$k_{x3} = v^{i} + \frac{\Delta t}{2} k_{v2}$,
$k_{x4} = v^{i} + k_{v3} \Delta t$, $k_{v1} = -\omega^2 x^{i}$,
$k_{v2} = -\omega^2 \left(x^i + \frac{\Delta t}{2} k_{x1}\right)$,
$k_{v3} = -\omega^2 \left(x^i + \frac{\Delta t}{2} k_{x2}\right)$,
$k_{x4} = -\omega^2 \left(x^{i} + k_{x3} \Delta t\right)$.

Импортируем необходимые модули.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

### Фазовая траектория маятника

Построим несколько периодов фазовой траектории системы.

In [ ]:
x0 = 0.5
v0 = 0.0

g = 9.81
l = 1
omega = math.sqrt(g/l)
T = 2*2*math.pi/omega # two periods
n_steps = 100
dt = T/n_steps

x = np.zeros(n_steps + 1)
v = np.zeros(n_steps + 1)
x[0] = x0
v[0] = v0
x_RK4 = np.zeros(n_steps + 1)
v_RK4 = np.zeros(n_steps + 1)
x_RK4[0] = x0
v_RK4[0] = v0

In [ ]:
for i in range(1, n_steps + 1):
    # Euler
    x[i] = x[i-1] + v[i-1] * dt
    v[i] = v[i-1] - omega ** 2 * x[i-1] * dt

    # Runge-Kutta 4th order
    kx1 = v_RK4[i-1]
    kv1 = -omega**2 * x_RK4[i-1]

    kx2 = v_RK4[i-1] + (dt/2.0) * kv1
    kv2 = -omega**2 * (x_RK4[i-1] + (dt/2.0) * kx1)

    kx3 = v_RK4[i-1] + (dt/2.0) * kv2
    kv3 = -omega**2 * (x_RK4[i-1] + (dt/2.0) * kx2)

    kx4 = v_RK4[i-1] + dt * kv3
    kv4 = -omega**2 * (x_RK4[i-1] + dt * kx3)

    x_RK4[i] = x_RK4[i-1] + dt/6.0*(kx1 + 2*kx2 + 2*kx3 + kx4)
    v_RK4[i] = v_RK4[i-1] + dt/6.0*(kv1 + 2*kv2 + 2*kv3 + kv4)

На следующей визуализации наглядно видно, как схема Эйлера накапливает ошибку.

In [ ]:
fig = plt.figure()
plt.plot(x, v, lw=2, c='blue')
plt.plot(x_RK4, v_RK4, lw=2, c='red')

plt.legend(['Euler', 'RK4'])
plt.xlabel('x')
plt.ylabel('x\'')

plt.show()

### Траектория матяника

Визуализируем анимацию траектории маятника.

$x$ - малый угол маятника, $l$ - длина маятника

In [ ]:
fig, ax = plt.subplots()
ax.set_xlim(-l*1.2, l*1.2)
ax.set_ylim(-l*1.2, l*0.1)
ax.set_aspect('equal')

line1, = ax.plot([], [], c='red', alpha=0.5)
line2, = ax.plot([0, l*math.sin(x0)], [0, -l*math.cos(x0)], 'o-', c='black', lw=2)

def animate(i, x):
    line1.set_data(l*np.sin(x[:i+1]), -l*np.cos(x[:i+1]))
    line2.set_data([0, l*math.sin(x[i])], [0, -l*math.cos(x[i])])
    return line1, line2,

ani = animation.FuncAnimation(fig, animate, frames=len(x), interval=75, fargs=(x_RK4,))

ani.save("pendulum1.gif")

from IPython.display import HTML
HTML(ani.to_jshtml())

#plt.show()

### Сравнение с аналитическим решением

Общее аналитическое решение уравнения $\ddot x + \omega^2 x = 0$:

$$x(t) = A \cos(t) + B \sin(t).$$

Для задачи Коши с начальными условиями

\begin{equation*}
\begin{cases}
x(0) = x_0,\\
\dot x(0) = 0.
\end{cases}
\end{equation*}

Решением будет:

$$x(t) = x_0 \cos(t).$$

In [ ]:
times = np.linspace(0.0, T, n_steps+1)
x_exact = x0*np.cos(omega*times)
v_exact = -x0*omega*np.sin(omega*times)

fig = plt.figure()
plt.plot(times, x_exact, c='black')
plt.plot(times, x, '--', c='blue')
plt.plot(times, x_RK4, '--', c='red')

plt.legend(['Exact', 'Euler', 'RK4'])
plt.xlabel('t')
plt.ylabel('x')
plt.title('Сравнение численных решений с точным')

plt.show()

Из графика видно, что схема Эйлера имеет большую ошибку, чем схема RK4.

### График энергии

Нарисуем график энергии для полученных численных решений и аналитического (энергия сохраняется, поэтому она равна константе).

Для данной системы $\ddot x + \omega^2 x = 0$ полная энергия равна:

$$E = \frac{1}{2} v^2 + \frac{1}{2} \omega^2 x^2.$$

In [ ]:
def energy(x, v):
    return v**2/2 + (omega*x)**2/2

exact_energy_values = energy(x_exact, v_exact)
Euler_energy_values = energy(x, v)
RK4_energy_values = energy(x_RK4, v_RK4)

fig = plt.figure()
plt.plot(times, exact_energy_values, c='black')
plt.plot(times, Euler_energy_values, '--', c='blue')
plt.plot(times, RK4_energy_values, '--', c='red')

plt.legend(['Exact', 'Euler', 'RK4'])

plt.show()

Схема Эйлера имеет большую ошибку и сильнее увеличивает энергию, чем схема RK4.

### Оценка скорости накопления ошибки

Оценим скорость накопления ошибки.

Для этого посчитаем массив из накопленых ошибок за все предыдущие шаги.

Предположим, что кумулятивная ошибка $\text{cumerror}$ асимпотически растёт как $t^p$ ($\text{cumerror} \propto С t^p$). Тогда, прологарифмировав это выражение получим $\log \text{cumerror} \approx p \log t + \log C$. Оценить $p$ можно с помощью метода наименьших квадратов, аппроксимируя прямую в координатах $\left(\log t, \log \text{cumerror}\right)$.


In [ ]:
error_x = np.abs(x - x_exact)
error_x_RK4 = np.abs(x_RK4 - x_exact)
error_E = np.abs(Euler_energy_values - exact_energy_values)
error_E_RK4 = np.abs(RK4_energy_values - exact_energy_values)

cumerror_x = np.cumsum(error_x)
cumerror_x_RK4 = np.cumsum(error_x_RK4)
cumerror_E = np.cumsum(error_E)
cumerror_E_RK4 = np.cumsum(error_E_RK4)

logtimes = np.log(times[1:])
logerror_x = np.log(cumerror_x[1:])
logerror_x_RK4 = np.log(cumerror_x_RK4[1:])
logerror_E = np.log(cumerror_E[1:])
logerror_E_RK4 = np.log(cumerror_E_RK4[1:])

from scipy.stats import linregress

slope_x, _, _, _, _ = linregress(logtimes, logerror_x)
slope_x_RK4, _, _, _, _ = linregress(logtimes, logerror_x_RK4)
slope_E, _, _, _, _ = linregress(logtimes, logerror_E)
slope_E_RK4, _, _, _, _ = linregress(logtimes, logerror_E_RK4)

fig, ax = plt.subplots(2, 1)
ax[0].grid(True)
ax[1].grid(True)
ax[0].loglog(times, cumerror_x, c='blue')
ax[0].loglog(times, cumerror_x_RK4, c='red')
ax[0].set_title("Накопленная ошибка по координате (логарифмическая шкала)")
ax[0].legend(['Euler, p={:.2f}'.format(slope_x), 'RK4, p={:.2f}'.format(slope_x_RK4)])
ax[1].loglog(times, cumerror_E, c='blue')
ax[1].loglog(times, cumerror_E_RK4, c='red')
ax[1].set_title("Накопленная ошибка по энергии (логарифмическая шкала)")
ax[1].legend(['Euler, p={:.2f}'.format(slope_E), 'RK4, p={:.2f}'.format(slope_E_RK4)])
fig.tight_layout()
plt.show()

## Продвинутый уровень

### Фазовый портрет

Визуализируем фазовый портрет системы. Для этого нарисуем полный период 6-ти фазовых тректорий с разными начальными условиями.

In [ ]:
trajectory_num = 6
x_max = 3.0
x0 = np.linspace(0.01, x_max, trajectory_num)
v0 = np.zeros(trajectory_num)

omega = 1
T = 2*math.pi/omega
n_steps = 100
dt = T/n_steps

x = np.zeros((trajectory_num, n_steps + 1))
v = np.zeros((trajectory_num, n_steps + 1))
x[:, 0] = x0
v[:, 0] = v0
x_RK4 = np.zeros((trajectory_num, n_steps + 1))
v_RK4 = np.zeros((trajectory_num, n_steps + 1))
x_RK4[:, 0] = x0
v_RK4[:, 0] = v0

Проводим расчёт для схемы Эйлера и схемы RK4:

In [ ]:
for i in range(1, n_steps + 1):
    # Euler
    x[:, i] = x[:, i-1] + v[:, i-1] * dt
    v[:, i] = v[:, i-1] - omega ** 2 * x[:, i-1] * dt

    # Runge-Kutta 4th order
    kx1 = v_RK4[:, i-1]
    kv1 = -omega**2 * x_RK4[:, i-1]

    kx2 = v_RK4[:, i-1] + (dt/2.0) * kv1
    kv2 = -omega**2 * (x_RK4[:, i-1] + (dt/2.0) * kx1)

    kx3 = v_RK4[:, i-1] + (dt/2.0) * kv2
    kv3 = -omega**2 * (x_RK4[:, i-1] + (dt/2.0) * kx2)

    kx4 = v_RK4[:, i-1] + dt * kv3
    kv4 = -omega**2 * (x_RK4[:, i-1] + dt * kx3)

    x_RK4[:, i] = x_RK4[:, i-1] + dt/6.0*(kx1 + 2*kx2 + 2*kx3 + kx4)
    v_RK4[:, i] = v_RK4[:, i-1] + dt/6.0*(kv1 + 2*kv2 + 2*kv3 + kv4)

In [ ]:
fig = plt.figure()
for i in range(trajectory_num):
    plt.plot(x[i, :], v[i, :], lw=2, c='blue')
    plt.plot(x_RK4[i, :], v_RK4[i, :], lw=2, c='red')

plt.legend(['Euler', 'RK4'])
plt.title('Фазовый портрет системы')
plt.xlabel('x')
plt.ylabel('x\'')

plt.show()

### Гармонически-сопряжённые маятники

Визуализируем систему гармонически-сопряженных маятников.

In [ ]:
omega = np.array([5.0/10, 6.0/10, 7.0/10, 8.0/10, 9.0/10, 1.0])
pendulum_num = len(omega)
x0 = np.full(pendulum_num, 1)
v0 = np.zeros(pendulum_num)

g = 9.81
T = 2*math.pi*10
l = g/omega**2
l_max = np.max(l)
n_steps = 200
dt = T/n_steps

x = np.zeros((pendulum_num, n_steps + 1))
v = np.zeros((pendulum_num, n_steps + 1))
x[:, 0] = x0
v[:, 0] = v0

In [ ]:
for i in range(1, n_steps + 1):
    kx1 = v[:, i-1]
    kv1 = -omega**2 * x[:, i-1]

    kx2 = v[:, i-1] + (dt/2.0) * kv1
    kv2 = -omega**2 * (x[:, i-1] + (dt/2.0) * kx1)

    kx3 = v[:, i-1] + (dt/2.0) * kv2
    kv3 = -omega**2 * (x[:, i-1] + (dt/2.0) * kx2)

    kx4 = v[:, i-1] + dt * kv3
    kv4 = -omega**2 * (x[:, i-1] + dt * kx3)

    x[:, i] = x[:, i-1] + dt/6.0*(kx1 + 2*kx2 + 2*kx3 + kx4)
    v[:, i] = v[:, i-1] + dt/6.0*(kv1 + 2*kv2 + 2*kv3 + kv4)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlim(-l_max*1.1, l_max*1.1)
ax.set_ylim(-l_max*1.1, l_max*0.1)
ax.set_aspect('equal')

line1 = []
line2 = []
color = ['red', 'orange', 'yellow', 'green', 'blue', 'purple'][::-1]
for i in range(pendulum_num):
    line, = ax.plot([], [], c=color[i], alpha=0.5)
    line1.append(line)
    line, = ax.plot([0, l[i]*np.sin(x0[i])], [0, -l[i]*np.cos(x0[i])], 'o-', c='black', lw=2)
    line2.append(line)

def animate(i, x):
    for j in range(pendulum_num):
        line1[j].set_data(l[j]*np.sin(x[j, :i+1]), -l[j]*np.cos(x[j, :i+1]))
        line2[j].set_data([0, l[j]*np.sin(x[j, i])], [0, -l[j]*np.cos(x[j, i])])
    return *line1, *line2

ani = animation.FuncAnimation(fig, animate, frames=n_steps+1, interval=50, fargs=(x,))

ani.save("pendulum.gif")

from IPython.display import HTML
HTML(ani.to_jshtml())
